In [1]:
#The script collects regridded files saved in scratch, calculates extreme precipitation days using 

#00. libraries and dask client 
import time
import xarray as xr
import numpy as np
import glob
import datetime
import time


import os
import gc

import matplotlib
import xarray, matplotlib.pyplot as plt
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm, LogNorm
import matplotlib as mpl


import dask 
import subprocess
import dask.array as da
from memory_profiler import profile
from filelock import FileLock
from dask.distributed import Client
from dask.utils import SerializableLock

# # Define a global or top-level lock (to reuse across function calls if needed)
# netcdf_write_lock = SerializableLock()


from dask.diagnostics import ProgressBar

client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45911,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41107,Total threads: 2
Dashboard: /proxy/46113/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:35827,


In [3]:
# ##P117 (used in paper&thesis ) -prepare data for Figure 3.4 /Figure 4.5, Figure A2-A4 and Figure B3-B5

##1.1 plot three 30 year changes --summarise data 
def summarize_periods_with_delta_to_netcdf(dataset_path_hist, dataset_path_future, variable_name,
                                           summaries_file, deltas_file):
    import xarray as xr
    import numpy as np
    import pandas as pd
    import cftime

    # Load both historical and future datasets
    ds1 = xr.open_dataset(dataset_path_hist, chunks={'time': -1, 'lat': 24, 'lon': 24})
    ds2 = xr.open_dataset(dataset_path_future, chunks={'time': -1, 'lat': 24, 'lon': 24})
    ds = xr.concat([ds1, ds2], dim='time')

    # Handle both datetime64 and cftime.datetime
    time_var = ds['time'].values
 
    # --- Handle cftime-based calendars (e.g. noleap, 360_day, gregorian) ---
    if isinstance(time_var[0], (cftime.DatetimeNoLeap,
                                cftime.Datetime360Day,
                                cftime.DatetimeGregorian,
                                cftime.DatetimeProlepticGregorian)):
        time_years = np.array([t.year for t in time_var])

    # --- Handle numpy datetime64 (both datetime64 and datetime64[ns]) ---
    elif np.issubdtype(time_var.dtype, np.datetime64):
        time_years = pd.to_datetime(time_var).year.values
    
    # --- Fallback: attempt to convert using pandas if type is mixed or unknown ---
    else:
        try:
            ds['time'] = pd.to_datetime(time_var)
            time_years = ds['time'].dt.year.values
        except Exception as e:
            raise ValueError(f"Unsupported or mixed time format: {type(time_var[0])}") from e

    periods = [(1981, 2010), (2036, 2065), (2071, 2100)]
    summaries = []
    deltas = []
    period_labels = []

    for start, end in periods:
        mask = (time_years >= start) & (time_years <= end)
        period_data = ds.isel(time=mask)
        summary = period_data[variable_name].sum(dim='time')
        summaries.append(summary)
        period_labels.append(f"{start}-01-01")

        if len(summaries) > 1:
            deltas.append(summary - summaries[0])  # delta relative to baseline 1981–2010

    # Save summaries
    summaries_da = xr.concat(summaries, dim='period')
    summaries_da['period'] = pd.to_datetime(period_labels)
    with FileLock(f"{summaries_file}.lock"):
        summaries_da.to_netcdf(summaries_file)

    # Save deltas
    if deltas:
        delta_labels = [f"{start}-01-01" for (start, _) in periods[1:]]
        deltas_da = xr.concat(deltas, dim='period')
        deltas_da['period'] = pd.to_datetime(delta_labels)
        deltas_da.to_netcdf(deltas_file)
        with FileLock(f"{deltas_file}.lock"):
            deltas_da.to_netcdf(deltas_file)


In [ ]:
#1.2.4 (used in paper&thesis)implement wbgt
gcms = ['ACCESS-ESM1-5','EC-Earth3-Veg','MPI-ESM1-2-HR','NorESM2-MM','UKESM1-0-LL']
rcms = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']#
variables = ['wbgt']
scenarios = ['ssp370','ssp126','ssp245']
base_dir = '/scratch/dt6/xw9650/tmp/'
#output_plot_dir = '/scratch/dt6/xw9650/tmp/threshold_maps'


for gcm in gcms:
    for rcm in rcms:
        for var in variables:
            for scen in scenarios:
                model_id = f"{var}_{gcm}_{rcm}_{scen}"
                hist_file = f"{base_dir}{var}_{gcm}_{rcm}_historical_univ_exceed.nc"
                fut_file = f"{base_dir}{var}_{gcm}_{rcm}_{scen}_univ_exceed.nc"
                summary_file = f"{base_dir}{model_id}_decadal_summary.nc"
                delta_file = f"{base_dir}{model_id}_decadal_delta.nc"

                print(f"\n🔄 Processing {model_id}...")

                try:
                    summarize_periods_with_delta_to_netcdf(
                        dataset_path_hist=hist_file,
                        dataset_path_future=fut_file,
                        variable_name=var,
                        summaries_file=summary_file,
                        deltas_file=delta_file
                    )
                    
                except Exception as e:
                    print(f"❌ Error processing {model_id}: {e}")


🔄 Processing wbgt_ACCESS-ESM1-5_NARCliM2-0-WRF412R3_ssp370...


In [1]:
#(used in paper &thesis )
#1.7 plot ensemble average change from historical to mid century to end century --all three scenarios 

def plot_median_exceedance_deltas_all_scenarios(summary_files_by_scenario, output_path):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import xarray as xr
    from matplotlib.colors import BoundaryNorm, ListedColormap
    import numpy as np
    import os
    from string import ascii_lowercase

    plt.rcParams.update({'font.size': 16})

    def get_delta_colormap_and_norm():
        boundaries = np.concatenate(([0, 4, 8, 16, 30], np.arange(60, 240, 30)))
        colors = [
            "#ffffe0", "#ffffb2", "#fed976", "#feb24c", "#fd8d3c",
            "#fc4e2a", "#e31a1c", "#bd0026", "#67000d", "#000000"
        ]
        cmap = ListedColormap(colors)
        norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)
        return cmap, norm

    def clean(var):
        return var.drop_vars(['height', 'quantile'], errors='ignore')

    scenario_label = {
        "ssp126": "SSP1-2.6",
        "ssp245": "SSP2-4.5",
        "ssp370": "SSP3-7.0",
    }

    land_mask = xr.open_dataset(
        '/scratch/dt6/xw9650/tmp/regridded/sftlf_AUS-18_ACCESS-ESM1-5_historical_r6i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_fx.nc'
    )['sftlf'] > 50

    delta_cmap, delta_norm = get_delta_colormap_and_norm()

    scenarios = ["ssp126", "ssp245", "ssp370"]
    fig, axes = plt.subplots(
        3, 2, figsize=(22, 22),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    

    for i, scen in enumerate(scenarios):
        summary_files_dict = summary_files_by_scenario.get(scen, {})
        delta_mid_list, delta_late_list = [], []
        model_ids = []

        for model_id, summary_file in summary_files_dict.items():
            try:
                summary_da = xr.open_dataarray(summary_file).where(land_mask)
                period_labels = {year: j for j, year in enumerate(summary_da['period'].dt.year.values)}

                base = clean(summary_da.isel(period=period_labels[1981]) / 30)
                mid  = clean(summary_da.isel(period=period_labels[2036]) / 30)
                late = clean(summary_da.isel(period=period_labels[2071]) / 30)

                delta_mid_list.append(mid - base)
                delta_late_list.append(late - base)
                model_ids.append(model_id)

            except Exception as e:
                print(f"❌ Skipping {model_id} ({scen}): {e}")

        if not delta_mid_list:
            continue

        median_delta_mid  = xr.concat(delta_mid_list, dim="model").median(dim="model")
        median_delta_late = xr.concat(delta_late_list, dim="model").median(dim="model")

        for j, data in enumerate([median_delta_mid, median_delta_late]):
            ax = axes[i, j]
            ax.coastlines()
            gl = ax.gridlines(draw_labels=True)
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (j == 0)
            gl.bottom_labels = (i == 2)

            title = "Δ Days/year (2036–2065 – 1981–2010)" if j == 0 else "Δ Days/year (2071–2100 – 1981–2010)"
            ax.set_title(title, fontsize=24, fontweight="bold", pad=8)

            img = data.plot(
                ax=ax, transform=ccrs.PlateCarree(),
                cmap=delta_cmap, norm=delta_norm, add_colorbar=False
            )

        # Row label (scenario)
        ax0 = axes[i, 0]
        ax0.text(
            -0.15, 0.5, scenario_label.get(scen, scen),
            transform=ax0.transAxes, rotation=90,
            va="center", ha="right", fontsize=30, fontweight="bold"
        )
        col_titles = ["Mid‑century (2036–2065)", "Late-century (2071-2100)"]
        for j, title in enumerate(col_titles):
            axes[0, j].set_title(title, fontsize=32, fontweight="bold", pad=12)

  
    # Shared colorbar
    cbar_ax = fig.add_axes([0.25, 0.06, 0.5, 0.02])
    # cbar = fig.colorbar(img, cax=cbar_ax, orientation='horizontal', label="Δ Days/year")
    boundaries = np.concatenate(([0, 4, 8, 16, 30], np.arange(60, 240, 30)))
    
    cbar = fig.colorbar(
        img,
        cax=cbar_ax,
        orientation='horizontal',
        ticks=boundaries,
        label="Δ Days/year"
    )
    

    cbar.ax.tick_params(labelsize=24, length=8, width=1.5)
    cbar.set_label("Δ Days/year", fontsize=24)


 
    plt.tight_layout(rect=[0, 0.08, 1, 0.96])

    # top-to-bottom, then left-to-right
    # for i, ax in enumerate(np.ravel(axes, order="F")):
    for i, ax in enumerate(axes.flat):   # or np.ravel(axes)
        ax.text(
            0.02, 0.98, f"{ascii_lowercase[i]})",   # a, b, c, ...
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=30, fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1.5)
        )

    plt.savefig(output_path, dpi=120)
    plt.close()
    print(f"✅ Saved median-change plot to {output_path}")


In [2]:
#(used in paper&thesis --wbgt)- Figure 6 / Figure 4.5 
# 1.7.4 CARS — combined plot for WBGT (all scenarios in one figure)

import os

gcms = ['ACCESS-ESM1-5','EC-Earth3-Veg','MPI-ESM1-2-HR','NorESM2-MM','UKESM1-0-LL']
rcms = ['NARCliM2-0-WRF412R3','NARCliM2-0-WRF412R5']
variables = ['wbgt']
scenarios = ['ssp126', 'ssp245', 'ssp370']  # all scenarios

base_dir = '/scratch/dt6/xw9650/tmp/'

summary_files_by_scenario = {}

for scen in scenarios:
    summary_files_dict = {}

    for gcm in gcms:
        for rcm in rcms:
            for var in variables:
                model_id = f"{var}_{gcm}_{rcm}_{scen}"
                summary_file = os.path.join(base_dir, f"{model_id}_decadal_summary.nc")
                if os.path.exists(summary_file):
                    summary_files_dict[model_id] = summary_file
                else:
                    print(f"⚠️ File not found: {summary_file}")

    if not summary_files_dict:
        print(f"⚠️ No summary files for scenario {scen}, skipping...")
        continue

    summary_files_by_scenario[scen] = summary_files_dict

# Output path for combined 3×2 panel
output_plot_path = os.path.join(
    base_dir, "threshold_maps",
    f"{variables[0]}_all_scenarios_multi_model_median_deltas_CARS.png"
)

# Call the new multi-scenario plot function
plot_median_exceedance_deltas_all_scenarios(summary_files_by_scenario, output_plot_path)


/jobfs/178103818.gadi-pbs/ipykernel_2584647/4153357481.py:127: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.08, 1, 0.96])


✅ Saved median-change plot to /scratch/dt6/xw9650/tmp/threshold_maps/wbgt_all_scenarios_multi_model_median_deltas_CARS.png


In [2]:
#(used in paper&thesis-supplymentary) 
#1.8.1 plot changes of exceedance by 30 years showing all ensembles 
def plot_all_models_exceedance_and_deltas(summary_files_dict, output_path):

    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import xarray as xr
    from matplotlib.colors import BoundaryNorm, ListedColormap
    import numpy as np

    plt.rcParams.update({'font.size': 18})

    # ---------------------------------------------------------
    # COLORMAP
    # ---------------------------------------------------------
    def get_delta_colormap_and_norm():
        boundaries = np.concatenate(([0, 4, 8, 16, 30], np.arange(60, 240, 30)))

        colors = [
            "#ffffe0", "#ffffb2", "#fed976", "#feb24c",
            "#fd8d3c", "#fc4e2a", "#e31a1c", "#bd0026",
            "#67000d", "#000000"
        ]

        cmap = ListedColormap(colors)
        norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)
        return cmap, norm

    def clean(var):
        return var.drop_vars(['height', 'quantile'], errors='ignore')

    # ---------------------------------------------------------
    # LAND MASK
    # ---------------------------------------------------------
    land_mask = xr.open_dataset(
        '/scratch/dt6/xw9650/tmp/regridded/sftlf_AUS-18_ACCESS-ESM1-5_historical_r6i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_fx.nc'
    )['sftlf'] > 50

    # ---------------------------------------------------------
    # LOAD DATA
    # ---------------------------------------------------------
    delta_mid_list = []
    delta_late_list = []
    model_ids = []

    for model_id, summary_file in summary_files_dict.items():
        try:
            da = xr.open_dataarray(summary_file).where(land_mask)

            period_labels = {y: i for i, y in enumerate(da['period'].dt.year.values)}

            base = clean(da.isel(period=period_labels[1981])) / 30
            mid = clean(da.isel(period=period_labels[2036])) / 30
            late = clean(da.isel(period=period_labels[2071])) / 30

            delta_mid_list.append(mid - base)
            delta_late_list.append(late - base)
            model_ids.append(model_id)

        except Exception as e:
            print(f"❌ Skipping {model_id}: {e}")

    # ---------------------------------------------------------
    # PARSE GCM / RCM STRUCTURE
    # ---------------------------------------------------------
    gcm_rcm_map = {}

    for i, mid in enumerate(model_ids):
        parts = mid.split('_')
        gcm = parts[1]
        rcm = parts[2]

        if gcm not in gcm_rcm_map:
            gcm_rcm_map[gcm] = {}

        gcm_rcm_map[gcm][rcm] = i

    gcm_list = list(gcm_rcm_map.keys())[:5]   # 5 rows (GCMs)

    # fix RCM ordering (consistent across all GCMs)
    rcm_list = sorted({rcm for g in gcm_rcm_map for rcm in gcm_rcm_map[g]})

    # ---------------------------------------------------------
    # FIGURE SETUP (5 × 4)
    # ---------------------------------------------------------
    nrows = len(gcm_list)
    ncols = 4

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(22, 5 * nrows),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    # Shared column headers
    fig.text(
        0.28, 0.965,               # centred over columns 0-1
        "2036–2065 − 1981–2010",
        ha='center',
        va='bottom',
        fontsize=24,
        fontweight='bold'
    )
    
    fig.text(
        0.72, 0.965,               # centred over columns 2-3
        "2071–2100 − 1981–2010",
        ha='center',
        va='bottom',
        fontsize=24,
        fontweight='bold'
    )


    delta_cmap, delta_norm = get_delta_colormap_and_norm()
    shared_img = None

    # ---------------------------------------------------------
    # MAIN LOOP (GCM × RCM × time)
    # ---------------------------------------------------------
    for i, gcm in enumerate(gcm_list):

        for j in range(ncols):

            ax = axes[i, j]

            rcm = rcm_list[j % 2]
            is_late = j >= 2

            if rcm not in gcm_rcm_map[gcm]:
                ax.axis("off")
                continue

            idx = gcm_rcm_map[gcm][rcm]

            data = delta_late_list[idx] if is_late else delta_mid_list[idx]

            ax.coastlines()
            gl = ax.gridlines(draw_labels=True)

            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (j == 0)
            gl.bottom_labels = (i == nrows - 1)

            img = data.plot(
                ax=ax,
                transform=ccrs.PlateCarree(),
                cmap=delta_cmap,
                norm=delta_norm,
                add_colorbar=False
            )

            if shared_img is None:
                shared_img = img

        

            # GCM + RCM label
            ax.text(
                0.03, 0.05,
                f"{gcm}\n{rcm}",
                transform=ax.transAxes,
                fontsize=16,
                fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='none',pad=2)
            )

    # ---------------------------------------------------------
    # COLORBAR
    # ---------------------------------------------------------
    cbar_ax = fig.add_axes([0.3, 0.06, 0.4, 0.02])

    boundaries = delta_norm.boundaries

    cbar = fig.colorbar(
        shared_img,
        cax=cbar_ax,
        orientation='horizontal',
        boundaries=boundaries,
        ticks=boundaries,
        spacing='uniform',
        label="Δ Days/year"
    )

    cbar.ax.tick_params(labelsize=20)

    # ---------------------------------------------------------
    # LAYOUT (Cartopy-safe)
    # ---------------------------------------------------------
    plt.subplots_adjust(
        left=0.05,
        right=0.98,
        top=0.95,
        bottom=0.12,
        wspace=0.05,
        hspace=0.15
    )

    plt.savefig(output_path, dpi=120)
    plt.close()

    print(f"✅ Saved GCM × RCM × MID/LATE matrix plot to {output_path}")

In [3]:
#(used in paper&thesis -supplimentry) --Figure S3-S5 /Figure B.3-B.5
#1.8.2 implement for wbgt
gcms = ['ACCESS-ESM1-5','EC-Earth3-Veg','MPI-ESM1-2-HR', 'NorESM2-MM','UKESM1-0-LL']
rcms = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
variables = ['wbgt']
scenarios = ['ssp370','ssp245','ssp126']
base_dir = '/scratch/dt6/xw9650/tmp/'
output_plot_path = f"/scratch/dt6/xw9650/tmp/threshold_maps/{variables[0]}_{scenarios[0]}_all_models_exceedance_summary_and_deltas.png"


for scen in scenarios:   
    summary_files_dict = {}
    for gcm in gcms:
        for rcm in rcms:
            for var in variables:
                model_id = f"{var}_{gcm}_{rcm}_{scen}"
                summary_file = os.path.join(base_dir, f"{model_id}_decadal_summary.nc")
                if os.path.exists(summary_file):
                    summary_files_dict[model_id] = summary_file
                else:
                    print(f"Warning: File not found {summary_file}")

    # set unique output per scenario
    output_plot_path = f"/scratch/dt6/xw9650/tmp/threshold_maps/{variables[0]}_{scen}_all_models_exceedance_summary_and_deltas.png"

    plot_all_models_exceedance_and_deltas(summary_files_dict, output_plot_path)



✅ Saved GCM × RCM × MID/LATE matrix plot to /scratch/dt6/xw9650/tmp/threshold_maps/wbgt_ssp370_all_models_exceedance_summary_and_deltas.png
✅ Saved GCM × RCM × MID/LATE matrix plot to /scratch/dt6/xw9650/tmp/threshold_maps/wbgt_ssp245_all_models_exceedance_summary_and_deltas.png
✅ Saved GCM × RCM × MID/LATE matrix plot to /scratch/dt6/xw9650/tmp/threshold_maps/wbgt_ssp126_all_models_exceedance_summary_and_deltas.png
